<a href="https://colab.research.google.com/github/The-Professor99/ML_Practice/blob/main/hands_on_learning/hands_on_ml_with_scikit-learn_Aurelien_textbook/Chapter_8_Dimensionality_Reduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Many ML problems involve thousands or even millions of features for each training instance. Not only do all these features make training extremely slow but they can also make it much harder to find a good solution. This is regarded as the curse of dimensionality.

Fortunately, in real world problems, it is often possible to reduce the number of features considerably, turning an intractable problem into a tractable one. For example, in the MNIST images, the pixels on the image borders are almost always white, so you could completely drop these pixels from the training set without losing much information. Additionally, two neighboring pixels are often highly correlated, if you merge them into a single pixel(eg, by taking the mean of the two pixel intensities), you will not lose too much information.

Reducing dimensionality does cause some information lost(just like compressing an image to JPEG can degrade its quality), so even though it will speed up training, it may make your system perform slightly worse. It also makes yyour pipelines a bit more complex and thus harder to maintain. So, if training is too slow, you should first try to train your system with the original dataset before considering using dimensionality reduction. In some cases, reducing dimensionality of the training data may filter out some noise and unnecessary details thus resulting in higher performance, but in general, it won't; it will just speed up training.

Apart from speeding up training, dimensionality reduction is also extremely useful for data visualization(or DataViz). Reducing the number of dimensions down to two or three makes it possible to plot a condensed vieew of a high dimensional training set on a graph and often gain some important insights by visually detecting patterns such as clusters. Moreover, DataViz is essential to communicate your conclusions to people who are not data scientists eg, decision makes who will use your results.

### The Curse of Dimensionality

There's plenty of space in high dimensions, as a result, high dimensional datasets are at risk of being very sparse: most training instances are likely to be far away from each other. This also means that a new instance will likely be far away from any training instance, making predictions much less reliable than in lower dimensions, since they will be based on much larger extrapolations. The more dimensions the training set has, the greater the risk of overfitting it.

In theory, one solution to the curse of dimensionality could be to increase the size of the training set to reach a sufficient density of training instances. Unfortunately, in practice, the number of training instances required to reach a given density grows exponentially with the number of dimensions.

Approaches for Dimensionality Reduction
1. Projection
2. Manifold Learning

### Projection

In most real-world problems, training instances are not spread out uniformly across all dimensions. Many features are almost constant, while others are highly correlated. As a result, all training instances lie within (or close to) a much lower-dimensional subspace of the high-dimensional space. Projection works by identifying the hyperplane that lies closest to the data, and then projecting the data onto it thereby reducing the dimensionality of the dataset from a higher dimension, to a lower dimension.
This approach may not always be best as in many cases, the subspace may twist and turn and that's where Manifold learning comes in.

### Manifold Learning

A 2D manifold is a 2D shape that can be bent and twisted in a higher dimensional space. Many dimensionality reduction algorithms work by modeling the manifold on which the training instances lie, this is called Manifold Learning. It relies on the Manifold Assumption(called Manifold Hypothesis) which holds that most real world high dimensional datasets lie close to a much lower dimensional manifold. This assumption is very often empirically observed.
The manifold assumption is also often accompanied by another implicit assumption: that the task at hand will be simpler if expressed in the lower dimensional space of the manifold. However, this implicit assumption does not always hold.

Reducing the dimensionality of your training set before training a model will usually speed up training, but it may not always lead to a better or simpler solution, it all depends on the dataset.

### PCA

The Principal Component Analysis algorithm first identifies the hyperplane that lies closest to the data, and then it projects the data onto it. But before you can project the training set onto a lower dimensional hyperplane, you first need to choose the right hyperplane. It seems reasonable to select the axis that preserves the maximum amount of variance, as it will most likely lose less information than the other projections.(see page 220, Figure 8.7). This choice is the axis that minimizes the mean squared distance between the original dataset and its projection onto that axis or it is the axis that maximizes the mean squared distance between the origin and the projected points.

### Principal Components

PCA identifies the axis that accounts for the largest amount of variance in the training set.It also finds a second axis, orthogonal to the first one, that accounts for the largest amount of remaining variance, if it were a higher dimensional dataset, PCA would also find a third axis, orthogonal to both previous axes, and a fourth, and a fifth, and so on, as many axes as the number of dimensions in the dataset.

The ith axis(that the dataset is projected to) is called the ith principal component(PC) of the data.

### Using Scikit Learn

Scikit-Learn's PCA class uses Singular value decomposition(SVD) to implement PCA. This class takes care of centering the data as PCA assumes the data is centered.

In [ ]:
from mpl_toolkits import mplot3d
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
np.random.seed(4)
m = 60
w1, w2 = 0.1, 0.3
noise = 0.1

angles = np.random.rand(m) * 3 * np.pi / 2 - 0.5
X = np.empty((m, 3))
X[:, 0] = np.cos(angles) + np.sin(angles)/2 + noise * np.random.randn(m) / 2
X[:, 1] = np.sin(angles) * 0.7 + noise * np.random.randn(m) / 2
X[:, 2] = X[:, 0] * w1 + X[:, 1] * w2 + noise * np.random.randn(m)

In [ ]:
%matplotlib

Using matplotlib backend: <object object at 0x7f07c424ab70>


In [ ]:
fig = plt.figure()
ax = plt.axes(projection='3d')

In [ ]:
D = pd.DataFrame(X)

In [ ]:
D

,0,1,2
0,-1.015700,-0.550913,-0.261326
1,-0.007717,0.599586,0.035078
2,-0.953171,-0.464537,-0.249203
3,-0.920123,0.210096,0.021824
4,-0.763097,0.158261,0.191525
5,1.118161,0.325087,0.317106
6,-1.022589,-0.643841,-0.133687
7,0.673520,-0.273425,-0.007878
8,1.016196,0.515466,0.467833
9,0.549577,0.677280,0.234016


In [ ]:
ax.scatter(D[0], D[1], D[2])

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components=2)
X2D = pca.fit_transform(X)

In [ ]:
C = pd.DataFrame(X2D)

In [ ]:
fig = plt.figure()

In [ ]:
plt.scatter(C[0], C[1])

In [ ]:
pca.components_.T[:, 0]

array([-0.93636116, -0.29854881, -0.18465208])

In [ ]:
pca.explained_variance_ratio_

array([0.84248607, 0.14631839])

### Choosing the Right Number of Dimensions

Instead of arbitrarily choosing the number of dimensions to reduce down to, it is simpler to choose the number of dimensions that add up to a sufficiently large portion of the variance. Unless, of course, you are reducing dimensionality for DataViz, in that case, you will want to reduce the dimensionality down to 2 or 3.

The ff code performs PCA indicating the ratio of variance(95%) we wish to preserve.

In [ ]:
pca = PCA(n_components=0.95)

In [ ]:
X_reduced = pca.fit_transform(X)

In [ ]:
pca.explained_variance_ratio_.sum()

0.8424860713752884

In [ ]:
# Or you can run pca, get the explained_variance_ratio_ list,
# get the point d at which the variance ratio is above
# a certain threshold and setting n_components=d and
# rerunning pca again
pca = PCA()
pca.fit(X)
cumsum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumsum >= 0.95) + 1

In [ ]:
# You can also plot the explained variance as a fxn
# of the number of dimensions and pick a value for d
# at the elbow point(where the explained variance
# stops growing rapidly.)
plt.plot(cumsum)

In [ ]:
import joblib

In [ ]:
mnist = joblib.load("datasets/mnist.pkl")

In [ ]:
X = mnist["data"]

In [ ]:
pca = PCA(n_components=0.95)

In [ ]:
X_reduced = pca.fit_transform(X)

In [ ]:
pd.DataFrame(X_reduced)

,0,1,2,3,4,5,6,7,8,9,...,144,145,146,147,148,149,150,151,152,153
0,122.255255,-316.233844,-51.131831,-556.897988,-49.210119,-217.069322,233.713325,188.824032,-358.798925,203.545001,...,6.090730,-49.152857,-24.712140,-12.328670,19.269423,43.061747,-34.361517,34.717035,-14.225757,21.382721
1,1010.494003,-289.963621,576.120745,-485.083384,-841.478355,-145.467158,-48.333388,-102.231088,34.786211,151.265038,...,14.777709,-65.033565,63.100130,-10.493406,19.344674,-24.773008,12.107962,23.878844,-6.542836,-24.902775
2,-58.995947,393.697445,-161.998184,529.220866,-313.159323,8.970184,-860.990143,374.289391,-78.501242,-192.717658,...,38.794609,-108.164813,28.292006,3.548508,87.941702,0.388307,87.041105,-5.362827,55.000209,-96.733971
3,-796.965019,-607.421250,295.522702,109.112831,25.666331,697.859150,-474.507968,-99.776449,298.602425,4.532776,...,-20.918722,-37.738248,12.602601,2.287826,43.344401,23.981709,3.015066,45.799165,-7.116732,-16.298993
4,-391.318921,729.804185,13.906964,-242.894182,98.772991,35.925878,-87.458146,-415.778351,200.297865,106.207817,...,7.899411,-6.794368,-10.995868,40.783493,9.521087,-29.210382,-6.491888,-60.877288,-44.692842,-17.404620
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,305.693874,-549.645167,-22.326828,277.944358,848.705944,925.506426,782.165977,276.077680,-546.250574,-29.464276,...,-56.189873,114.301516,8.411661,18.721368,-14.614459,-62.815449,-23.631555,-9.965013,-12.049371,88.505095
69996,569.186045,-699.027039,-834.350404,-158.989310,94.290795,-191.454882,672.682817,784.064380,-48.606593,-38.252319,...,-6.328271,-4.024638,23.564925,113.461011,-0.660043,-61.719770,4.376064,-12.166449,27.519573,-102.757747
69997,-271.507013,590.078500,341.368869,-434.994569,673.650548,139.689240,-39.122005,-344.735041,-52.187667,-803.531591,...,-22.030756,1.879233,-2.520168,-59.815240,-49.888549,8.159636,-28.581627,-43.757147,35.782160,49.966128
69998,-310.224823,-116.727151,635.719997,-282.508931,-155.263739,-275.917806,154.831162,144.836501,285.018359,-205.063623,...,40.426347,-24.370045,-35.446581,6.260216,31.776088,18.589848,-38.354402,-21.863453,20.401528,-42.682775


In [ ]:
pca = PCA()
pca.fit(X)
cumsum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumsum >= 0.95) + 1

In [ ]:
plt.plot(cumsum)

In [ ]:

import matplotlib as mpl

In [ ]:
def show_digit_image_and_class(X, y, index, isArray=False):
    plt.figure()
    if isArray:
        some_digit = X[index]
        some_digit_image = some_digit.reshape(28, 28)
    else:
        some_digit = X.iloc[index]
        some_digit_image = some_digit.values.reshape(28, 28)
    plt.imshow(some_digit_image, cmap= mpl.cm.binary, interpolation="nearest")
    plt.axis("off")
    plt.colorbar()
    plt.show()
    print("Corresponding y class is: ", y.iloc[index])

### PCA for Compression

After compression, it is also possible to decompress the reduced dataset back to it's original dimensions by applying the inverse transformation of the PCA project. this won't give back the original data since the projection lost a bit of information but it will likely be close to the original data. The mean squared distance between the original data and the reconstructed data is called the reconstruction error

In [ ]:
pca = PCA(n_components = 154)
X_reduced = pca.fit_transform(X)
X_recovered = pca.inverse_transform(X_reduced)

In [ ]:
# shows original digit and the same digit
# after compression and decompression
index =9945
show_digit_image_and_class(X, mnist["target"], index)
show_digit_image_and_class(X_recovered, mnist["target"], index, isArray=True)

Corresponding y class is:  5
Corresponding y class is:  5


### Incremental PCA

One problem with preceding implementation of PCA is that they require the whole training set to fit in memory in order for the algorithm to run. Incremental PCA(IPCA) algorithms allow you to split the training set into mini batches and feed an IPCA algorithm one mini-batch at a time. This is useful for large training sets and for applying PCA online.
The ff code splits MNIST dataset into 100 mini batches and feeds them to sklearn's IPCA class to reduce the dimensionality down to 154 dimensions

In [ ]:
from sklearn.decomposition import IncrementalPCA

In [ ]:
n_batches = 100
inc_pca = IncrementalPCA(n_components=154)
for X_batch in np.array_split(X, n_batches):
    inc_pca.partial_fit(X_batch)

X_reduced = inc_pca.transform(X)

In [ ]:
X_reduced

array([[ 122.25500694,  316.23364499,   51.13178161, ...,   79.54978737,
          70.81789108,    9.17132981],
       [1010.49445865,  289.96446993, -576.12081754, ...,   24.44785951,
         -50.0768693 ,   39.40749108],
       [ -58.99599856, -393.69804681,  161.99724511, ...,   56.43654621,
         -87.9226803 ,  -17.8323958 ],
       ...,
       [-271.50686236, -590.07806446, -341.36905325, ...,  -46.64576339,
          35.14735446,  -25.86730048],
       [-310.22486483,  116.72752672, -635.71984162, ...,  -42.34733241,
         -46.03059498,  -19.6249888 ],
       [1058.86192623,   83.39244069, -731.34206351, ...,  -54.49489283,
         -89.19509602,  -24.05562367]])

### Kernel PCA

This algorithm applies the kernel trick to PCA, making it possible to perform complex nonlinear projections for dimensionality reduction. It is often good at preserving clusters of instances after projections, or sometimes, even unrolling datasets that lie close to a twisted manifold.

In [ ]:
def plot_swiss_roll(pca, X, y):
    plt.figure()
    X_reduced = pca.fit_transform(X)

    plt.plot()
    plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=t, cmap=plt.cm.hot)
    plt.xlabel("$z_1$", fontsize=18)
    plt.ylabel("$z_2$", fontsize=18, rotation=0)
    plt.grid(True)

In [ ]:
from sklearn.decomposition import KernelPCA
from sklearn.datasets import make_swiss_roll

In [ ]:
X, t = make_swiss_roll(n_samples=1000, noise=0.2, random_state=42)

In [ ]:
# swiss roll plot
fig = plt.figure()
ax = fig.add_subplot(projection="3d")
ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=t, cmap=plt.cm.hot)

In [ ]:
rbf_pca = KernelPCA(n_components=2, kernel="rbf", gamma=0.04)

In [ ]:
X_reduced = rbf_pca.fit_transform(X)

In [ ]:
plot_swiss_roll(rbf_pca, X, t)

kPCA is an unsupervised learning algorithm hence there is no obvious performance measure to help you select the best  kernel and hyperparameter values. However, dimensionality reduction is often a preparation step for a supervised learning task so you can use grid search to select the kernel and hyperparameters that lead to the best performance on that task.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

In [ ]:
clf = Pipeline([
    ("kpca", KernelPCA(n_components=2)),
    ("log_reg", LogisticRegression())
])
param_grid = [{
    "kpca__gamma": np.linspace(0.03, 0.05, 10),
    "kpca__kernel": ["rbf", "sigmoid"]
}]

In [ ]:
y = t > 6.9
grid_search = GridSearchCV(clf, param_grid, cv=3)
grid_search.fit(X, y)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('kpca', KernelPCA(n_components=2)),
                                       ('log_reg', LogisticRegression())]),
             param_grid=[{'kpca__gamma': array([0.03      , 0.03222222, 0.03444444, 0.03666667, 0.03888889,
       0.04111111, 0.04333333, 0.04555556, 0.04777778, 0.05      ]),
                          'kpca__kernel': ['rbf', 'sigmoid']}])

In [ ]:
print(grid_search.best_params_)

{'kpca__gamma': 0.043333333333333335, 'kpca__kernel': 'rbf'}


### LLE

Locally Linear Embedding is a manifold learning technique that does not rely on projections. It works by first measuring how each training instance linearly relates to its closest neighbors, and then looking for a low dimensional representation of the training set where these relationships are best preserved. This approach makes it particularly good at unrolling twisted manifolds, especially when there is not too much noise.

In [ ]:
from sklearn.manifold import LocallyLinearEmbedding

In [ ]:
lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10)
X_reduced = lle.fit_transform(X)

In [ ]:
plot_swiss_roll(lle, X, t)

In [ ]:
lle.

In [ ]:
ax

[]